In [1]:
import os
import pathlib

# Application packages
from tabulate import tabulate
import netCDF4

# stage_in packages
from pystac import Catalog

# stage_out packages
from datetime import datetime, timezone
from pystac import CatalogType, Item, Asset
from pystac.layout import TemplateLayoutStrategy

## Metadata Cell

The below cell is tagged as the 'metadata' cell. This defines metadata values that are exposed by the OGC Application Package created. The keywords show below all have default values if they are not present in the cell. But it is encourage to define custom values for each application like shown below.

In [2]:
# Identification of the application
id = "mdps-example-application"
doc = "Example of using mdps-app-generator to produce an OGC Application pacakge"
label = "Example MDPS OGC Application"

# Information on the application
author = "James McDuffie"
citation = "https://github.com/unity-sds/mdps-example-application"
codeRepository = "https://github.com/unity-sds/mdps-example-application"
keywords = "mdps, ogc, example"
license = "Apache Version 2"
version = "1.1.0"
softwareVersion = "1.1.0"
releaseNotes = ""

## Parameters Cell

The below cell is tagged as the 'parameter' cell. This enables us to overwrite the below values at runtime. There are some special values in the below cell.

* `input_stac_catalog_dir` has the `# type: stage-in` annotation. This variable tells your algorithm the name of the directory where to read inputs from. At run time it will be populated with a STAC catalog and the files it references. The application generator process will connect your variable name to the special `input` parameter in automatically generated CWL files.
* `output_stac_catalog_dir` has the `# type: stage-out` annotation. This is a directory where you write ALL of your output files along with a STAC catalog that references files you would like to persist outside of the algorithm run. The application generator process will connect your variable name to the special `output` parameter in automatically generated CWL files.

In [3]:
input_stac_catalog_dir  = 'test/stage_in/'         # type: stage-in
output_stac_catalog_dir = 'test/process_results/'  # type: stage-out

# Filename written to the working directory
summary_table_filename = "summary_table.txt"

# For eventual cataloging of this file in the unity environment
output_collection = "mdps-example-application___1"

# Examples of optional arbitrary arguments of different data types
example_argument_int = 1
example_argument_float = 1.0
example_argument_string = "string"
example_argument_bool = True
example_argument_empty = None # type: string Allow a null value or a string


In [4]:
input_catalog = os.path.join(input_stac_catalog_dir, "catalog.json")
print("reading {}".format(input_catalog))

reading test/stage_in/catalog.json


# Output Example Arguments

Table useful for debugging connection of arguments from CWL into notebook. Used for unity-app-generator development and as an example, not an application requirement.

In [5]:
argument_names = [
    'example_argument_int',
    'example_argument_float',
    'example_argument_string',
    'example_argument_bool',
    'example_argument_empty',
]

table_data = []
column_names = ['argument_name', 'type', 'value']
for arg_name in argument_names:
    arg_val = locals()[arg_name]
    table_data.append( (arg_name, type(arg_val), arg_val) )

tabulate(table_data, headers=column_names, tablefmt='html')

argument_name,type,value
example_argument_int,<class 'int'>,1
example_argument_float,<class 'float'>,1.0
example_argument_string,<class 'str'>,string
example_argument_bool,<class 'bool'>,True
example_argument_empty,<class 'NoneType'>,


# Import Files from STAC Item Collection

Load filenames from the stage_in STAC item collection file

In [6]:
inp_catalog = Catalog.from_file(input_catalog)
inp_catalog.make_all_asset_hrefs_absolute()

data_filenames = []
for item in inp_catalog.get_items(recursive=True):
    fn = os.path.relpath(item.assets['data'].href)
    data_filenames.append(fn)

data_filenames

['test/stage_in/SNDR.SS1330.CHIRP.20160822T0005.m06.g001.L1_AQ.std.v02_48.G.200425095850.nc',
 'test/stage_in/SNDR.SS1330.CHIRP.20160822T0011.m06.g002.L1_AQ.std.v02_48.G.200425095901.nc']

# Summary Table

AKA the result of this "application"

In [7]:
column_names = [
    'product_name',
    'product_name_type_id',
    'shortname',
    'product_version',
    'date_created',
    'time_coverage_start',
    'time_coverage_end',
    'geospatial_lat_mid',
    'geospatial_lon_mid',
]

In [8]:
table_data = []
for data_file in data_filenames:
    nc_file = netCDF4.Dataset(data_file, "r")
    
    table_row = []
    for col_name in column_names:
        table_row.append(getattr(nc_file, col_name))
    table_data.append(table_row)

In [9]:
# Output the table in html format
tabulate(table_data, headers=column_names, tablefmt='html')

product_name,product_name_type_id,shortname,product_version,date_created,time_coverage_start,time_coverage_end,geospatial_lat_mid,geospatial_lon_mid
SNDR.SS1330.CHIRP.20160822T0005.m06.g001.L1_AQ.std.v02_48.G.200425095850.nc,L1_AQ,SNDR13CHRP1,v02.48.00,2021-04-25T05:59:08Z,2016-08-22T00:05:22Z,2016-08-22T00:11:22Z,-48.6062,12.4563
SNDR.SS1330.CHIRP.20160822T0011.m06.g002.L1_AQ.std.v02_48.G.200425095901.nc,L1_AQ,SNDR13CHRP1,v02.48.00,2021-04-25T05:59:19Z,2016-08-22T00:11:22Z,2016-08-22T00:17:22Z,-69.3979,-1.98753


In [10]:
# Write the table in text format
pathlib.Path(output_stac_catalog_dir).mkdir(parents=True, exist_ok=True)
output_filename = os.path.join(output_stac_catalog_dir, summary_table_filename)
with open(output_filename, "w") as summary_file:
    summary_file.write(tabulate(table_data, headers=column_names))

# Create stage-out item catalog

In [11]:
# Create a collection
out_catalog  = Catalog(id=output_collection, description="mdps-example-application STAC catalog")
    
# Create a Item for the collection
item = Item(
    id=summary_table_filename, 
    collection=out_catalog.id,
    geometry=None,
    bbox=None,
    datetime=datetime.now(timezone.utc),
    properties={},
)
out_catalog.add_item(item)

# Add output file(s) to the dataset
asset = Asset(
    href=summary_table_filename, 
    title="csv file",
    roles=["data"]
)
item.add_asset(key=os.path.basename(summary_table_filename), asset=asset)

# Add the dataset to the collection
strategy = TemplateLayoutStrategy(item_template="")
out_catalog.normalize_hrefs(output_stac_catalog_dir, strategy=strategy)
out_catalog.save(catalog_type=CatalogType.SELF_CONTAINED, dest_href=output_stac_catalog_dir)